# Transmission Line Data Analysis (Fall 2026)

In [1]:
# Libraries
from sys import platform

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
# XGBoost
import xgboost as xgb
# mlp model
import torch.nn as nn
import torch.optim as optim
# tcn
import torch.nn.functional as F



# cuda

# import torch

#print(platform.python_version())
#print(torch.__version__)
#print(torch.cuda.is_available())
#print(torch.cuda.get_device_name(0))


In [ ]:
### Data Loading

df = pd.read_csv("stage4_features.csv")


df.head()

In [ ]:
metadata = [
    "scenario_id", "split", "class_name"]

In [ ]:
### Data Preprocessing

# 2. Drop columns safely (ignoring them if they don't exist in the current df)
cols_to_drop = ['scenario_id']
df.drop(columns=cols_to_drop, errors='ignore', inplace=True)

df.head()

['class_name', 'split', 
                     'send_v_rms_ratio_a', 'send_v_rms_ratio_b', 'send_v_rms_ratio_c', 
                     'send_i_rms_ratio_a', 'send_i_rms_ratio_b', 'send_i_rms_ratio_c', 
                     'send_v0_rms_v', 'send_v1_rms_v', 'send_v2_rms_v', 
                     'send_i0_rms_a', 'send_i1_rms_a', 'send_i2_rms_a', 
                     'send_v0_v1_ratio', 'send_v2_v1_ratio', 'send_i0_i1_ratio', 
                     'send_i2_i1_ratio', 'send_active_power_w', 'send_reactive_power_var', 
                     'send_power_factor', 'send_z_magnitude_a_ohm', 'send_z_magnitude_b_ohm', 
                     'send_z_magnitude_c_ohm', 'send_v_rms_imbalance_pct', 'send_i_rms_imbalance_pct', 
                     'receive_v_rms_a_v', 'receive_v_rms_b_v', 'receive_v_rms_c_v', 
                     'receive_i_rms_a_a', 'receive_i_rms_b_a', 'receive_i_rms_c_a', 
                     'receive_v_rms_ratio_a', 'receive_v_rms_ratio_b', 'receive_v_rms_ratio_c', 
                     'receive_i_rms_ratio_a', 'receive_i_rms_ratio_b', 'receive_i_rms_ratio_c', 
                     'receive_v0_rms_v', 'receive_v1_rms_v', 'receive_v2_rms_v', 
                     'receive_i0_rms_a', 'receive_i1_rms_a', 'receive_i2_rms_a', 
                     'receive_v0_v1_ratio', 'receive_v2_v1_ratio', 'receive_i0_i1_ratio', 
                     'receive_i2_i1_ratio', 'receive_active_power_w', 'receive_reactive_power_var', 
                     'receive_power_factor', 'receive_z_magnitude_a_ohm', 'receive_z_magnitude_b_ohm', 
                     'receive_z_magnitude_c_ohm', 'receive_v_rms_imbalance_pct', 'receive_i_rms_imbalance_pct', 
                     'send_v_spectral_entropy_mean', 'send_v_spectral_entropy_max', 'send_v_fundamental_residual_ratio_mean', 
                     'send_v_crest_factor_mean', 'send_v_flatline_fraction_mean', 'send_i_spectral_entropy_mean', 
                     'send_i_spectral_entropy_max', 'send_i_fundamental_residual_ratio_mean', 'send_i_crest_factor_mean', 
                     'send_i_flatline_fraction_mean', 'receive_v_spectral_entropy_mean', 
                     'receive_v_spectral_entropy_max', 'receive_v_fundamental_residual_ratio_mean', 
                     'receive_v_crest_factor_mean', 'receive_v_flatline_fraction_mean', 'receive_i_spectral_entropy_mean', 
                     'receive_i_spectral_entropy_max', 'receive_i_fundamental_residual_ratio_mean', 'receive_i_crest_factor_mean', 
                     'receive_i_flatline_fraction_mean']

In [ ]:
# Data Analysis
# List features variance
X = df.drop(columns=['class_name', 'split', 
                     'send_v_rms_ratio_a', 'send_v_rms_ratio_b', 'send_v_rms_ratio_c', 
                     'send_i_rms_ratio_a', 'send_i_rms_ratio_b', 'send_i_rms_ratio_c', 
                     'send_power_factor', 'send_active_power_w', 'send_reactive_power_var',
                     'receive_v_rms_a_v', 'receive_v_rms_b_v', 'receive_v_rms_c_v', 
                     'receive_i_rms_a_a', 'receive_i_rms_b_a', 'receive_i_rms_c_a', 
                     'receive_v_rms_ratio_a', 'receive_v_rms_ratio_b', 'receive_v_rms_ratio_c', 
                     'receive_i_rms_ratio_a', 'receive_i_rms_ratio_b', 'receive_i_rms_ratio_c', 
                     'receive_v0_rms_v', 'receive_v1_rms_v', 'receive_v2_rms_v', 
                     'receive_i0_rms_a', 'receive_i1_rms_a', 'receive_i2_rms_a', 
                     'receive_v0_v1_ratio', 'receive_v2_v1_ratio', 'receive_i0_i1_ratio', 
                     'receive_i2_i1_ratio', 'receive_active_power_w', 'receive_reactive_power_var', 
                     'receive_power_factor', 'receive_z_magnitude_a_ohm', 'receive_z_magnitude_b_ohm', 
                     'receive_z_magnitude_c_ohm', 'receive_v_rms_imbalance_pct', 'receive_i_rms_imbalance_pct', 
                     'send_v_spectral_entropy_mean', 'send_v_spectral_entropy_max', 'send_v_fundamental_residual_ratio_mean', 
                     'send_v_crest_factor_mean', 'send_v_flatline_fraction_mean', 'send_i_spectral_entropy_mean', 
                     'send_i_spectral_entropy_max', 'send_i_fundamental_residual_ratio_mean', 'send_i_crest_factor_mean', 
                     'send_i_flatline_fraction_mean', 'receive_v_spectral_entropy_mean', 
                     'receive_v_spectral_entropy_max', 'receive_v_fundamental_residual_ratio_mean', 
                     'receive_v_crest_factor_mean', 'receive_v_flatline_fraction_mean', 'receive_i_spectral_entropy_mean', 
                     'receive_i_spectral_entropy_max', 'receive_i_fundamental_residual_ratio_mean', 'receive_i_crest_factor_mean', 
                     'receive_i_flatline_fraction_mean'], errors='ignore')

y = df['class_name']

X.var().sort_values(ascending=False)




### Model Testing

In [ ]:
# Regression

train = df["split"].eq("train")
validation = df["split"].eq("validation")
test = df["split"].eq("test")

X_train, y_train = X.loc[train], y.loc[train]
X_val, y_val = X.loc[validation], y.loc[validation]
X_test, y_test = X.loc[test], y.loc[test]

print(X.columns.tolist())

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(solver='lbfgs', max_iter=200)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

# 6. Evaluate the model
print("Accuracy Score:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


In [ ]:
for split_name, X_split, y_split in [
    ("validation", X_val_scaled, y_val),
    ("test", X_test_scaled, y_test),
    ("ood_test", scaler.transform(X.loc[df["split"].eq("ood_test")]), y.loc[df["split"].eq("ood_test")])
]:
    y_pred = model.predict(X_split)
    print(split_name, accuracy_score(y_split, y_pred))
    print(classification_report(y_split, y_pred))

In [ ]:
# Random Forest Classifier
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    min_samples_leaf=1,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)
rf_y_pred = rf_model.predict(X_test)


accuracy = accuracy_score(y_test, rf_y_pred)
print("Random Forest Classifier Accuracy:", accuracy)
print("\nConfusion Matrix:\n", confusion_matrix(y_test, rf_y_pred))
print("\nClassification Report:\n", classification_report(y_test, rf_y_pred))

In [ ]:
for split_name, X_split, y_split in [
    ("validation", X_val, y_val),
    ("test", X_test, y_test),
    ("ood_test", X.loc[df["split"].eq("ood_test")], y.loc[df["split"].eq("ood_test")])
]:
    y_pred = rf_model.predict(X_split)
    print(split_name, accuracy_score(y_split, y_pred))
    print(classification_report(y_split, y_pred))

In [ ]:
from sklearn.model_selection import GridSearchCV

# Perform a grid search for hyperparameter tuning and test ood
param_grid = {
    "n_estimators": [100, 200, 500],
    "max_depth": [4, 8, 16, None],
    "min_samples_leaf": [1, 5, 10, 20],
    "max_features": ["sqrt", "log2"]
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(
        random_state=42,
        n_jobs=-1
    ),
    param_grid=param_grid,
    scoring="accuracy",
    cv=5,
    n_jobs=-1,
    verbose=2
)

grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best CV accuracy:", grid_search.best_score_)

best_rf_model = grid_search.best_estimator_

for split_name, X_split, y_split in [
    ("validation", X_val, y_val),
    ("test", X_test, y_test),
    ("ood_test", X.loc[df["split"].eq("ood_test")], y.loc[df["split"].eq("ood_test")])
]:
    y_pred = best_rf_model.predict(X_split)
    print("\n" + split_name)
    print("Accuracy:", accuracy_score(y_split, y_pred))
    print(classification_report(y_split, y_pred))



## XGradient boost


In [ ]:
# XGBoost Classifier
xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

# convert labels to integers for XGBoost
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)

# Fit the model using encoded labels
xgb_model.fit(X_train, y_train_encoded)

# Evaluate on the splits
for split_name, X_split, y_split in [
    ("validation", X_val, y_val),
    ("test", X_test, y_test),
    ("ood_test", X.loc[df["split"].eq("ood_test")], y.loc[df["split"].eq("ood_test")])
]:
    # 1. Predict the integer codes
    y_pred_encoded = xgb_model.predict(X_split)
    
    # 2. Convert the integer predictions back to original labels
    y_pred = label_encoder.inverse_transform(y_pred_encoded)
    
    # 3. Safely compare with original labels (y_split)
    print(f"--- {split_name.upper()} ---")
    print("Accuracy:", accuracy_score(y_split, y_pred))
    print(classification_report(y_split, y_pred))


In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset

# MLP classifier

torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Encode labels using the existing encoder
y_train_mlp = label_encoder.transform(y_train)
y_val_mlp = label_encoder.transform(y_val)
y_test_mlp = label_encoder.transform(y_test)

train_dataset = TensorDataset(
    torch.tensor(X_train_scaled, dtype=torch.float32),
    torch.tensor(y_train_mlp, dtype=torch.long)
)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)

mlp_model = nn.Sequential(
    nn.Linear(X_train_scaled.shape[1], 128),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(64, len(label_encoder.classes_))
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(mlp_model.parameters(), lr=1e-3)

# Training
for epoch in range(100):
    mlp_model.train()
    for features, labels in train_loader:
        features, labels = features.to(device), labels.to(device)

        optimizer.zero_grad()
        loss = criterion(mlp_model(features), labels)
        loss.backward()
        optimizer.step()

# Evaluation
def evaluate_mlp(X_data, y_data, split_name):
    mlp_model.eval()
    with torch.no_grad():
        features = torch.tensor(X_data, dtype=torch.float32).to(device)
        predictions = mlp_model(features).argmax(dim=1).cpu().numpy()

    predictions = label_encoder.inverse_transform(predictions)
    print(f"\n{split_name}")
    print("Accuracy:", accuracy_score(y_data, predictions))
    print(classification_report(y_data, predictions))

evaluate_mlp(X_val_scaled, y_val, "Validation")
evaluate_mlp(X_test_scaled, y_test, "Test")

ood_mask = df["split"].eq("ood_test")
evaluate_mlp(
    scaler.transform(X.loc[ood_mask]),
    y.loc[ood_mask],
    "OOD Test"
)

# 

In [ ]:
# Temporal Convolutional Network (TCN)
class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super().__init__()
        self.chomp_size = chomp_size

    def forward(self, x):
        return x[:, :, :-self.chomp_size] if self.chomp_size else x


class TemporalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dilation, dropout=0.2):
        super().__init__()
        padding = (3 - 1) * dilation

        self.net = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, 3,
                      padding=padding, dilation=dilation),
            Chomp1d(padding),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(out_channels, out_channels, 3,
                      padding=padding, dilation=dilation),
            Chomp1d(padding),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.residual = (
            nn.Conv1d(in_channels, out_channels, 1)
            if in_channels != out_channels else nn.Identity()
        )

    def forward(self, x):
        return F.relu(self.net(x) + self.residual(x))


class TCNClassifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.network = nn.Sequential(
            TemporalBlock(1, 32, dilation=1),
            TemporalBlock(32, 64, dilation=2),
            TemporalBlock(64, 64, dilation=4)
        )
        self.classifier = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.network(x)
        x = x.mean(dim=2)
        return self.classifier(x)


# Convert feature vectors to [samples, channels, sequence_length]
X_train_tcn = torch.tensor(
    X_train_scaled[:, None, :], dtype=torch.float32
)
X_val_tcn = torch.tensor(
    X_val_scaled[:, None, :], dtype=torch.float32
)
X_test_tcn = torch.tensor(
    X_test_scaled[:, None, :], dtype=torch.float32
)

ood_mask = df["split"].eq("ood_test")
X_ood_tcn = torch.tensor(
    scaler.transform(X.loc[ood_mask])[:, None, :],
    dtype=torch.float32
)

y_train_tcn = torch.tensor(y_train_mlp, dtype=torch.long)
y_val_tcn = torch.tensor(y_val_mlp, dtype=torch.long)
y_test_tcn = torch.tensor(y_test_mlp, dtype=torch.long)
y_ood_tcn = torch.tensor(
    label_encoder.transform(y.loc[ood_mask]),
    dtype=torch.long
)

tcn_dataset = TensorDataset(X_train_tcn, y_train_tcn)
tcn_loader = DataLoader(tcn_dataset, batch_size=128, shuffle=True)

tcn_model = TCNClassifier(
    num_classes=len(label_encoder.classes_)
).to(device)

tcn_optimizer = optim.Adam(tcn_model.parameters(), lr=1e-3)
tcn_criterion = nn.CrossEntropyLoss()

for epoch in range(100):
    tcn_model.train()

    for features, labels in tcn_loader:
        features = features.to(device)
        labels = labels.to(device)

        tcn_optimizer.zero_grad()
        predictions = tcn_model(features)
        loss = tcn_criterion(predictions, labels)
        loss.backward()
        tcn_optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1}/100, Loss: {loss.item():.6f}")


def evaluate_tcn(X_data, y_data, split_name):
    tcn_model.eval()

    with torch.no_grad():
        predictions = tcn_model(X_data.to(device)).argmax(dim=1)
        predictions = predictions.cpu().numpy()

    predictions = label_encoder.inverse_transform(predictions)
    actual = label_encoder.inverse_transform(y_data.numpy())

    print(f"\n{split_name}")
    print("Accuracy:", accuracy_score(actual, predictions))
    print(classification_report(actual, predictions))


evaluate_tcn(X_val_tcn, y_val_tcn, "Validation")
evaluate_tcn(X_test_tcn, y_test_tcn, "Test")
evaluate_tcn(X_ood_tcn, y_ood_tcn, "OOD Test")

# WITH NEW SPLITS

In [2]:
features_path = "stage4_features.csv"
manifest_path = "stage4_bulk_manifest.csv"

features = pd.read_csv(features_path)
manifest = pd.read_csv(manifest_path)

# Keep only needed metadata from manifest
meta = manifest[
    [
        "scenario_id",
        "class_name",
        "operating_domain_id",
        "is_ood"
    ]
]

# Merge domain info into features
df = features.merge(
    meta,
    on=["scenario_id", "class_name"],
    how="left"
)

# Check for merge problems
if df["operating_domain_id"].isna().any():
    missing = df[df["operating_domain_id"].isna()]
    raise ValueError(f"Missing domain info for {len(missing)} rows")

# Define new grouped split
def assign_split(domain_id):
    if 1 <= domain_id <= 12:
        return "train"
    elif 13 <= domain_id <= 15:
        return "validation"
    elif 16 <= domain_id <= 18:
        return "test"
    elif 19 <= domain_id <= 20:
        return "ood_test"
    else:
        return "unknown"

df["split_grouped"] = df["operating_domain_id"].apply(assign_split)

# Optional: replace old split
df["split"] = df["split_grouped"]
df = df.drop(columns=["split_grouped"])

# Save corrected dataset
df.to_csv("stage4_features_grouped_split.csv", index=False)

print(df["split"].value_counts())
print(pd.crosstab(df["split"], df["class_name"]))
print(pd.crosstab(df["split"], df["operating_domain_id"]))

split
train         6600
validation    1650
test          1650
ood_test      1100
Name: count, dtype: int64
class_name   AB  ABC  ABG   AG   BC  BCG   BG   CA  CAG   CG  Healthy
split                                                                
ood_test    100  100  100  100  100  100  100  100  100  100      100
test        150  150  150  150  150  150  150  150  150  150      150
train       600  600  600  600  600  600  600  600  600  600      600
validation  150  150  150  150  150  150  150  150  150  150      150
operating_domain_id   1    2    3    4    5    6    7    8    9    10   11  \
split                                                                        
ood_test               0    0    0    0    0    0    0    0    0    0    0   
test                   0    0    0    0    0    0    0    0    0    0    0   
train                550  550  550  550  550  550  550  550  550  550  550   
validation             0    0    0    0    0    0    0    0    0    0    0   

ope

In [3]:
df = pd.read_csv("stage4_features_grouped_split.csv")

target_col = "class_name"

drop_cols = [
    "scenario_id",
    "split",
    "class_name",
    "operating_domain_id",
    "is_ood"
]

feature_cols = [c for c in df.columns if c not in drop_cols]

train_df = df[df["split"] == "train"]
val_df = df[df["split"] == "validation"]
test_df = df[df["split"] == "test"]
ood_df = df[df["split"] == "ood_test"]

X_train = train_df[feature_cols]
y_train = train_df[target_col]

X_val = val_df[feature_cols]
y_val = val_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]

X_ood = ood_df[feature_cols]
y_ood = ood_df[target_col]

In [4]:
# 

# Random Forest Classifier
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    min_samples_leaf=1,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)
rf_y_pred = rf_model.predict(X_test)

for split_name, X_split, y_split in [
    ("validation", X_val, y_val),
    ("test", X_test, y_test),
    ("ood_test", X_ood, y_ood),
]:
    y_pred = rf_model.predict(X_split)
    print(split_name, accuracy_score(y_split, y_pred))
    print(classification_report(y_split, y_pred))

validation 1.0
              precision    recall  f1-score   support

          AB       1.00      1.00      1.00       150
         ABC       1.00      1.00      1.00       150
         ABG       1.00      1.00      1.00       150
          AG       1.00      1.00      1.00       150
          BC       1.00      1.00      1.00       150
         BCG       1.00      1.00      1.00       150
          BG       1.00      1.00      1.00       150
          CA       1.00      1.00      1.00       150
         CAG       1.00      1.00      1.00       150
          CG       1.00      1.00      1.00       150
     Healthy       1.00      1.00      1.00       150

    accuracy                           1.00      1650
   macro avg       1.00      1.00      1.00      1650
weighted avg       1.00      1.00      1.00      1650

test 1.0
              precision    recall  f1-score   support

          AB       1.00      1.00      1.00       150
         ABC       1.00      1.00      1.00       150
